# Day G2 — 생성 모델의 원리: 분포를 배우고 새 점을 뽑기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/G-ai-biomed/day-G2.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/G-ai-biomed/day-G2.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — PCA를 "선형 오토인코더"로 써서 새 신호 만들기

In [ ]:
import numpy as np
from sklearn.decomposition import PCA

rng = np.random.default_rng(0)
# 가상 신호 80개: 진폭·위상·기울기가 조금씩 다른 걸음 주기 파형(100칸) + 잡음
t = np.linspace(0, 2 * np.pi, 100)
amp = rng.normal(1.0, 0.15, 80)
phase = rng.normal(0.0, 0.3, 80)
slope = rng.normal(0.0, 0.1, 80)
X = amp[:, None] * np.sin(t + phase[:, None]) + slope[:, None] * t / (2 * np.pi)
X = X + rng.normal(0, 0.05, X.shape)

# 인코더 = 100칸 → 3개 숫자(잠재 공간), 디코더 = 3개 숫자 → 100칸
pca = PCA(n_components=3).fit(X)
Z = pca.transform(X)
X_rec = pca.inverse_transform(Z)
print("설명된 분산 비율:", np.round(pca.explained_variance_ratio_, 3))
print("복원 오차(RMSE):", round(float(np.sqrt(((X - X_rec) ** 2).mean())), 4))

# 생성: 잠재 공간의 분포(평균·공분산)를 추정해 새 점을 뽑고 디코더로 펼친다
Z_new = rng.multivariate_normal(Z.mean(axis=0), np.cov(Z.T), size=5)
X_new = pca.inverse_transform(Z_new)
print("실제 신호 최댓값 범위:", np.round([X.max(1).min(), X.max(1).max()], 2))
print("새로 만든 신호 5개의 최댓값:", np.round(X_new.max(1), 2))

# 가장 가까운 실제 신호와의 거리: 0에 가까우면 '복사', 너무 크면 '엉뚱함'
d = np.sqrt(((X_new[:, None, :] - X[None, :, :]) ** 2).mean(-1)).min(1)
print("새 신호 → 가장 가까운 실제 신호 RMSE:", np.round(d, 3))

### 예시 2 — 확산의 앞 과정과 가장 단순한 자기회귀 생성

In [ ]:
import numpy as np
from collections import Counter, defaultdict

rng = np.random.default_rng(0)
# (1) 확산 모델의 '앞 과정': 원래 신호에 잡음을 조금씩 섞으면 결국 순수 잡음이 된다
t = np.linspace(0, 2 * np.pi, 100)
x0 = np.sin(t)
print("단계  원 신호와의 상관")
x = x0.copy()
for step in range(1, 51):
    beta = 0.05                                    # 한 단계에 섞는 잡음의 양
    x = np.sqrt(1 - beta) * x + np.sqrt(beta) * rng.normal(size=x.shape)
    if step in (1, 5, 10, 25, 50):
        print(f"{step:>3}   {np.corrcoef(x0, x)[0, 1]:.3f}")
# 확산 모델은 이 과정을 '거꾸로' 한 단계씩 되돌리는 법을 신경망으로 배운다(여기서는 생략).

# (2) 자기회귀(autoregressive) 생성: 앞 글자를 보고 다음 글자를 확률로 고른다
train = ["MKTAYIAK", "MKTLLVAG", "MKSAYLAK", "MKTAYLVG", "MRTAYIAK", "MKTAFIAG"]  # 가상 펩타이드
nxt = defaultdict(Counter)
for s in train:
    s = "^" + s + "$"                              # 시작·끝 표시
    for a, b in zip(s, s[1:]):
        nxt[a][b] += 1

def sample(max_len=12):
    out, cur = "", "^"
    while len(out) < max_len:
        chars, counts = zip(*nxt[cur].items())
        cur = rng.choice(chars, p=np.array(counts) / sum(counts))
        if cur == "$":
            break
        out += cur
    return out

gen = [sample() for _ in range(8)]
print("생성된 서열:", gen)
print("학습 서열을 그대로 복사한 개수:", sum(g in train for g in gen), "/", len(gen))

### 예시 3 — PyTorch로 쓴 아주 작은 VAE

> PyTorch가 필요합니다. 과정의 conda 환경이라면 `conda install -n neurobio -c conda-forge pytorch`로 설치합니다. 출력은 PyTorch 2.14(conda-forge)에서 실행한 그대로입니다.

In [ ]:
import torch, torch.nn as nn

class TinyVAE(nn.Module):
    def __init__(self, d_in=100, d_z=3):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(d_in, 32), nn.ReLU())
        self.mu, self.logvar = nn.Linear(32, d_z), nn.Linear(32, d_z)    # 점 대신 '구름'
        self.dec = nn.Sequential(nn.Linear(d_z, 32), nn.ReLU(), nn.Linear(32, d_in))

    def forward(self, x):
        h = self.enc(x)
        mu, logvar = self.mu(h), self.logvar(h)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)          # 구름에서 한 점 뽑기
        return self.dec(z), mu, logvar

def vae_loss(x, x_hat, mu, logvar):
    recon = ((x - x_hat) ** 2).sum(dim=1).mean()                        # 복원 오차
    kl = (-0.5 * (1 + logvar - mu ** 2 - logvar.exp()).sum(dim=1)).mean()  # 구름을 표준정규 쪽으로
    return recon + kl

# 사용 예: 가상 데이터 300개 × 100차원(3개 잠재 요인 + 잡음)으로 200번 학습
torch.manual_seed(0)
x = torch.randn(300, 3) @ torch.randn(3, 100) + 0.1 * torch.randn(300, 100)
vae = TinyVAE()
opt = torch.optim.Adam(vae.parameters(), lr=1e-2)
for step in range(201):
    x_hat, mu, logvar = vae(x)
    loss = vae_loss(x, x_hat, mu, logvar)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0:
        print(f"step {step:3d}  loss {loss.item():.1f}")
with torch.no_grad():
    new = vae.dec(torch.randn(5, 3))       # 잠재 공간에서 5점을 뽑아 새 샘플 만들기
print(new.shape)